# Data exploration — Rogers Pass snow profiles, period 2025-2026

Extension of the published dataset (v1, DOI 10.20383/103.01634) with the March 2026 campaign
(2026-03-03 to 2026-03-10). Read-only inspection of `raw_data/2025-2026/`, plus a cross-period
comparison with the published v1 files in `published/v1/`.

Summary and findings: `artifacts/2025-2026/data_exploration.md`.

In [1]:
import re
import zipfile
import warnings
from collections import Counter
from pathlib import Path

import openpyxl
import pandas as pd
from snowmicropyn import Profile

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

DATASET = Path("..").resolve()          # notebook runs from notebooks/
RAW = DATASET / "raw_data" / "2025-2026"
V1 = DATASET / "published" / "v1"
assert RAW.is_dir() and V1.is_dir(), "raw_data/2025-2026 or published/v1 missing"

DAY_DATES = {
    "Day1_RoundHill_03-03-2026": "2026-03-03",
    "Day2_Fidelity_RH_04-03-2026": "2026-03-04",
    "DAY3_RoundHill Drone and Rogers Pass_05-03-2026": "2026-03-05",
    "DAY4_JimBay-RoundHill-GopherButte_20260306": "2026-03-06",
    "DAY5_RestDay_20260307": "2026-03-07",
    "DAY6_ReposeEtFilm_20260308": "2026-03-08",
    "DAY7_HermitMeadows_20260309": "2026-03-09",
    "DAY8_Fidelity_20260310": "2026-03-10",
}


def rel(p: Path) -> str:
    return p.relative_to(RAW).as_posix()


def day_of(p: Path) -> str:
    return p.relative_to(RAW).parts[0]

## 1. File inventory

Every file is classified by instrument / content class. Classes follow the scope agreed at Step 1:
out-of-scope classes are hazard assessment PDFs, field notebook photos, logistics and blank templates.

In [2]:
def classify(p: Path) -> str:
    parts = [s.lower() for s in p.relative_to(RAW).parts]
    name, ext = p.name.lower(), p.suffix.lower()
    if parts[0] == "templates":
        return "template (excluded)"
    if ext in (".heic", ".jpg", ".jpeg", ".png"):
        return "photo (excluded)"
    if ext == ".docx":
        return "field readme / notes"
    if ext == ".pdf":
        return "hazard assessment PDF (excluded)"
    if "dku" in parts:
        return "radar dual-Ku"
    if "radar_k" in parts or "ka" in parts:
        return "radar K-band"
    if "smp" in parts and ext == ".pnt":
        return "SMP"
    if "snowscope" in parts and ext == ".csv":
        return "SnowScope"
    if "iris" in parts or ext == ".txt" and "iris" in name:
        return "IRIS"
    if ext in (".kml", ".kmz") or "gps" in name or any("gps" in s for s in parts[1:-1]) and ext == ".csv":
        return "GNSS"
    if ext == ".xlsx" and "spatial" in name:
        return "spatial linkage workbook"
    if ext == ".xlsx":
        return "stratigraphy workbook"
    if ext == ".docx":
        return "field readme / notes"
    return "other"


files = sorted(p for p in RAW.rglob("*") if p.is_file())
inv = pd.DataFrame({
    "path": [rel(p) for p in files],
    "day": [day_of(p) for p in files],
    "ext": [p.suffix.lower() for p in files],
    "size": [p.stat().st_size for p in files],
    "cls": [classify(p) for p in files],
})
print(len(inv), "files")
print(inv.groupby("cls").agg(files=("path", "size"), bytes=("size", "sum")).sort_values("files", ascending=False))

567 files
                                  files      bytes
cls                                               
radar dual-Ku                       175  190210899
radar K-band                        141   15436448
SnowScope                            89    2917834
photo (excluded)                     57  146000371
SMP                                  48   39019520
stratigraphy workbook                14    3931440
IRIS                                 13      24004
hazard assessment PDF (excluded)      9    2994320
field readme / notes                  8      60359
GNSS                                  6     127689
spatial linkage workbook              4      30078
template (excluded)                   3     577611


In [3]:
print(pd.crosstab(inv["day"], inv["cls"]).to_string())
print("\nUnclassified:", inv.loc[inv.cls == "other", "path"].tolist())
print("Empty files:", inv.loc[inv["size"] == 0, "path"].tolist())

cls                                              GNSS  IRIS  SMP  SnowScope  field readme / notes  hazard assessment PDF (excluded)  photo (excluded)  radar K-band  radar dual-Ku  spatial linkage workbook  stratigraphy workbook  template (excluded)
day                                                                                                                                                                                                                                                     
DAY3_RoundHill Drone and Rogers Pass_05-03-2026     0     3    0         10                     1                                 1                 8             0              0                         0                      2                    0
DAY4_JimBay-RoundHill-GopherButte_20260306          1     3   16         27                     1                                 1                10            23             54                         1                      3                    0
DAY6

## 2. Stratigraphy workbooks (StratiTemplate)

Sheets present, `AVY profile` header fields (date, time, elevation, aspect, incline, location, observer),
and filled data rows per sheet. v1 workbooks had only `AVY profile`, `Stability Tests`, `Density`, `IRIS`.

In [4]:
def nonempty_rows(ws, start=1):
    return [r for r in ws.iter_rows(min_row=start, values_only=True) if any(v not in (None, "") for v in r)]


def header_fields(ws) -> dict:
    """Return label -> value pairs found in the first 6 rows of an AVY profile sheet."""
    out = {}
    for row in ws.iter_rows(max_row=6, values_only=True):
        vals = list(row)
        for i, v in enumerate(vals):
            if isinstance(v, str) and v.strip().upper() in {"DATE", "TIME", "ELEVATION", "ASPECT", "INCLINE", "LOCATION", "OBSERVER", "SNOW PROFILE"}:
                nxt = next((x for x in vals[i + 1:] if x not in (None, "")), None)
                out[v.strip().upper()] = nxt
    return out


strat = [p for p in files if classify(p) == "stratigraphy workbook"]
rows, sheet_rows = [], []
for p in strat:
    wb = openpyxl.load_workbook(p, read_only=True, data_only=True)
    h = header_fields(wb["AVY profile"]) if "AVY profile" in wb.sheetnames else {}
    rows.append({"file": p.name, "day": day_of(p)[:4], **h})
    for ws in wb.worksheets:
        sheet_rows.append({"file": p.name, "sheet": ws.title, "rows": len(nonempty_rows(ws))})
hdr = pd.DataFrame(rows)
print(len(strat), "stratigraphy workbooks")
print(hdr.to_string())

14 stratigraphy workbooks
                                                 file   day                  SNOW PROFILE          OBSERVER                 DATE       TIME ELEVATION   ASPECT                    INCLINE       LOCATION
0                                      RoundHill.xlsx  Day1                          ORG.             JM-AL  2026-03-03 00:00:00      12H37       SKY     EAST  SITE CARACT.,\nOBJECTIVES            NaN
1                         uppersnowpackstability.xlsx  Day1                          ORG.                MC  2026-03-03 00:00:00      13h45    2040.0        E                       15.0            NaN
2                     FidelityStation_2023-03-04.xlsx  Day2                          ORG.      MCD,AL,VP,NA  2026-03-04 00:00:00     1100.0    1905.0  INCLINE  SITE CARACT.,\nOBJECTIVES            NaN
3                         GopherButte_2026-03-04.xlsx  Day2                          ORG.        MC, KH, JM  2026-03-04 00:00:00   11:27:00    1930.0        W            

In [5]:
template = openpyxl.load_workbook(RAW / "templates" / "StratiTemplate.xlsx", read_only=True, data_only=True)
template_rows = {ws.title: len(nonempty_rows(ws)) for ws in template.worksheets}
sr = pd.DataFrame(sheet_rows)
sr["template_rows"] = sr["sheet"].map(template_rows)
sr["filled"] = sr["rows"] > sr["template_rows"].fillna(0)
print(sr.pivot_table(index="file", columns="sheet", values="rows", fill_value="").to_string())
print("\nTemplate rows per sheet:", template_rows)
empty = sr.groupby("file")["filled"].sum()
print("\nWorkbooks with no sheet filled beyond the template:", empty[empty == 0].index.tolist())

sheet                                             AVY profile   BHG Density Dual KU  GPS  IRIS IRIS_3 Radar K Radar K GPS RadarK   SS SnowScope Snowscope Stability Tests   dku  smp
file                                                                                                                                                                                
2026-03-09_Hermit_SurfacePit1_StratiTemplate.xlsx        24.0  23.0    53.0     1.0        8.0            0.0                     0.0                                 9.0        2.0
2026-03-09_Hermit_SurfacePit2_StratiTemplate.xlsx         8.0   1.0    43.0     1.0        8.0            0.0                     0.0                                 1.0        2.0
20260305_RogersPass.xlsx                                 48.0  42.0    49.0               16.0   16.0                             5.0                                 3.0        5.0
20260305_Round Hill Upper Snowpack.xlsx                  26.0  20.0    46.0               15.0 

C:\Users\PaulBillecocq\AppData\Local\Temp\ipykernel_10036\4019421220.py:6: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  print(sr.pivot_table(index="file", columns="sheet", values="rows", fill_value="").to_string())


In [6]:
v1_strat = sorted((V1 / "snow_stratigraphy").glob("*.xlsx"))
v1_sheets = Counter(s for p in v1_strat for s in openpyxl.load_workbook(p, read_only=True).sheetnames)
new_sheets = Counter(r["sheet"] for r in sheet_rows)
print("v1 sheets:", dict(v1_sheets))
print("2025-2026 sheets:", dict(new_sheets))
print("Sheet names only in 2025-2026:", sorted(set(new_sheets) - set(v1_sheets)))

v1 sheets: {'AVY profile': 7, 'Stability Tests': 7, 'Density': 7, 'IRIS': 7}
2025-2026 sheets: {'AVY profile': 14, 'BHG': 11, 'Stability Tests': 14, 'Density': 12, 'IRIS': 14, 'smp': 10, 'dku': 3, 'RadarK': 1, 'SS': 9, 'Radar K GPS': 1, 'Snowscope': 1, 'SnowScope': 1, 'IRIS_3': 1, 'Dual KU': 5, 'Radar K': 5, 'GPS': 1}
Sheet names only in 2025-2026: ['BHG', 'Dual KU', 'GPS', 'IRIS_3', 'Radar K', 'Radar K GPS', 'RadarK', 'SS', 'SnowScope', 'Snowscope', 'dku', 'smp']


In [7]:
def layer_columns(ws, max_scan=15):
    """Return the first row that looks like the layer-table header in an AVY profile sheet."""
    for row in ws.iter_rows(max_row=max_scan, values_only=True):
        vals = [str(v).strip() for v in row if v not in (None, "")]
        if any(v.upper().startswith(("HEIGHT", "DEPTH", "HS")) for v in vals) and len(vals) >= 4:
            return tuple(vals)
    return None


cols_new = Counter(layer_columns(openpyxl.load_workbook(p, read_only=True, data_only=True)["AVY profile"]) for p in strat)
cols_v1 = Counter(layer_columns(openpyxl.load_workbook(p, read_only=True, data_only=True)["AVY profile"]) for p in v1_strat)
print("2025-2026 AVY profile layer headers:")
for k, v in cols_new.items(): print(f"  {v} x {k}")
print("v1 AVY profile layer headers:")
for k, v in cols_v1.items(): print(f"  {v} x {k}")

2025-2026 AVY profile layer headers:
  1 x ('ASPECT', 'EAST', 'INCLINE', 'SITE CARACT.,\nOBJECTIVES', 'HS 344')
  1 x ('ASPECT', 'E', 'INCLINE', '15.0', 'HS', '310.0', 'SITE CARACT.,\nOBJECTIVES')
  11 x ('HEIGHT', 'RESISTANCE', 'FORM', 'EXTENT', 'Ѳ (LWC)', 'WEIGHT', 'DENSITY', 'COMMENTS & TEST DATA', 'TEMPERATURE')
  1 x ('HEIGHT', 'RESISTANCE', 'BHG', 'FORM', 'Ѳ (LWC)', 'IRIS Depth', 'Irish Value', 'COMMENTS & TEST DATA', 'TEMPERATURE')
v1 AVY profile layer headers:
  7 x ('HEIGHT', 'RESISTANCE', 'FORM', 'EXTENT', 'Ѳ (LWC)', 'WEIGHT', 'DENSITY', 'COMMENTS & TEST DATA', 'TEMPERATURE')


## 3. Instrument linkage (workbook sheets and spatial workbooks)

The linkage sheets and spatial workbooks map profile positions or transect points to instrument file
identifiers. Each referenced identifier is checked against the files on disk for the same day.

In [8]:
def ids_on_disk(day: str) -> dict:
    """Instrument identifiers present on disk for one day folder."""
    d = RAW / day
    dku = {int(p.name[:4]) for p in d.rglob("*") if p.is_file() and "dku" in p.parts and p.name[:4].isdigit()}
    radk = {int(p.name[:4]) for p in d.rglob("*.txt") if ("Radar_K" in p.parts or "ka" in p.parts) and p.name[:4].isdigit()}
    smp = {int(p.stem[-4:]) for p in d.rglob("*.pnt")}
    ss = {int(m.group(1)) for p in d.rglob("*.csv") for m in [re.search(r"Profile(\d+)", p.name)] if m}
    return {"Dual KU": dku, "Radar K": radk, "SMP": smp, "SS": ss}


def to_int(v):
    try:
        return int(float(str(v).strip()))
    except (TypeError, ValueError):
        return None


link_rows = []
for p in [p for p in files if classify(p) == "spatial linkage workbook"]:
    ws = openpyxl.load_workbook(p, read_only=True, data_only=True).worksheets[0]
    data = nonempty_rows(ws)
    head = [str(v).strip() if v is not None else "" for v in data[0]]
    disk = ids_on_disk(day_of(p))
    for col in ["Dual KU", "Radar K", "SMP", "SS"]:
        if col not in head:
            continue
        i = head.index(col)
        refs = [to_int(r[i]) for r in data[1:] if i < len(r) and to_int(r[i]) is not None]
        missing = sorted(set(refs) - disk[col])
        link_rows.append({"workbook": p.name, "column": col, "refs": len(refs), "missing_on_disk": len(missing), "examples": missing[:6]})
    print(p.name, "| columns:", head, "| points:", len(data) - 1)
print()
print(pd.DataFrame(link_rows).to_string())

Spatial_Round_Hill.xlsx | columns: ['#', 'GPS', 'Dual KU', 'Radar K', 'SMP', 'SS', 'HS1', 'HS2', 'HS3'] | points: 21
JimBay_Spatial.xlsx | columns: ['#', 'GPS', 'Dual KU', 'Radar K', 'SMP', 'SS', 'HS'] | points: 18
20260309_HermitMeadow_Spatial.xlsx | columns: ['#', 'GPS', 'Radar K - 1', 'Radar K - 2', 'Radar K - 3', 'SS', 'HS - 1', 'HS - 2', 'HS - 3'] | points: 24
20260310_RoundHill_Spatial.xlsx | columns: ['#', 'GPS', 'Radar K', 'Dual KU', 'HS', '', '', '', ''] | points: 18

                              workbook   column  refs  missing_on_disk    examples
0              Spatial_Round_Hill.xlsx  Dual KU    18                0          []
1              Spatial_Round_Hill.xlsx  Radar K    18                0          []
2              Spatial_Round_Hill.xlsx      SMP    18                0          []
3              Spatial_Round_Hill.xlsx       SS    18                0          []
4                  JimBay_Spatial.xlsx  Dual KU    15                2  [846, 860]
5                  J

In [9]:
sheet_link = []
for p in strat:
    wb = openpyxl.load_workbook(p, read_only=True, data_only=True)
    disk = ids_on_disk(day_of(p))
    for sheet, key in [("dku", "Dual KU"), ("Dual KU", "Dual KU"), ("smp", "SMP"), ("SS", "SS"), ("Snowscope", "SS"), ("SnowScope", "SS"), ("Radar K", "Radar K"), ("RadarK", "Radar K")]:
        if sheet not in wb.sheetnames:
            continue
        nums = [to_int(v) for r in nonempty_rows(wb[sheet], start=2) for v in r if to_int(v) is not None and to_int(v) >= 0]
        cand = [n for n in nums if n in disk[key] or n > 60]   # crude filter: ids, not angles/depths
        missing = sorted({n for n in cand if n not in disk[key]})
        sheet_link.append({"workbook": p.name, "sheet": sheet, "ids_found": len(set(cand)), "missing_on_disk": missing[:10]})
print(pd.DataFrame(sheet_link).to_string())

                                             workbook      sheet  ids_found                                    missing_on_disk
0                                      RoundHill.xlsx        dku         10  [95, 780, 781, 782, 783, 784, 785, 786, 787, 789]
1                                      RoundHill.xlsx        smp          6                                              [170]
2                                      RoundHill.xlsx         SS          0                                                 []
3                                      RoundHill.xlsx     RadarK          0                                                 []
4                         uppersnowpackstability.xlsx        smp          0                                                 []
5                     FidelityStation_2023-03-04.xlsx        dku         14                                              [818]
6                     FidelityStation_2023-03-04.xlsx        smp          6                                    

## 4. IRIS text files

In [10]:
def read_iris(p: Path) -> pd.DataFrame:
    df = pd.read_csv(p, header=None, skipinitialspace=True)
    df.columns = ["date", "time", "label", "voltage"][: df.shape[1]] if df.shape[1] == 4 else ["time", "voltage"]
    return df


iris = []
for p in [p for p in files if classify(p) == "IRIS"]:
    df = read_iris(p)
    iris.append({"file": rel(p), "day": day_of(p)[:4], "cols": df.shape[1], "rows": len(df), "dates": sorted(df["date"].unique().tolist()) if "date" in df else None,
                 "v_min": df["voltage"].min(), "v_max": df["voltage"].max(), "labels": sorted(df["label"].unique().tolist()) if "label" in df else None})
print(pd.DataFrame(iris).to_string())
v1_iris = sorted((V1 / "iris").glob("*"))
print("\nv1 IRIS example:", v1_iris[0].name, "columns:", read_iris(v1_iris[0]).shape[1])

                                                                                                                 file   day  cols  rows       dates  v_min  v_max  labels
0                                                       Day1_RoundHill_03-03-2026/IRIS/IRIS2_20260303_fullProfile.TXT  Day1     4    60  [2026-3-3]  0.182  2.014  [IRIS]
1                                            Day1_RoundHill_03-03-2026/IRIS/IRIS2_20260303_upperSnowpackStability.TXT  Day1     4    18  [2026-3-3]  0.749  0.940  [IRIS]
2                                                Day2_Fidelity_RH_04-03-2026/IRIS/IRIS#1-Gopher Butte TP/20260304.TXT  Day2     2    40        None  0.096  1.768    None
3                                                       Day2_Fidelity_RH_04-03-2026/IRIS/IRIS#2-RH/IRIS2_20260304.TXT  Day2     4    78  [2026-3-4]  0.178  2.035  [IRIS]
4                                          Day2_Fidelity_RH_04-03-2026/IRIS/IRIS#3-FidelityStation/IRIS3_20260304.TXT  Day2     2   126        None  0

## 5. SnowMicroPen (`.pnt`, read with `snowmicropyn`)

In [11]:
smp = []
for p in [p for p in files if classify(p) == "SMP"]:
    prof = Profile.load(p)
    lat, lon = prof.coordinates if prof.coordinates else (None, None)
    smp.append({"file": rel(p), "day": day_of(p)[:4], "timestamp": prof.timestamp, "lat": lat, "lon": lon,
                "samples": len(prof.samples), "depth_mm": round(prof.samples.distance.max(), 1),
                "smp_serial": prof.smp_serial, "force_max_N": round(prof.samples.force.max(), 2)})
smp = pd.DataFrame(smp)
print(len(smp), "SMP profiles")
print(smp.groupby("day").agg(n=("file", "size"), first=("timestamp", "min"), last=("timestamp", "max"),
                             depth_max=("depth_mm", "max"), no_gps=("lat", lambda s: s.isna().sum())).to_string())
print("\nSerials:", smp.smp_serial.value_counts().to_dict())
print("Sample count range:", smp.samples.min(), "-", smp.samples.max())

Latitude value -99999.0 invalid, replacing by None (file S35M0736)


Longitude value None invalid, replacing by None (file S35M0736)


Latitude value -99999.0 invalid, replacing by None (file S35M0782)


Longitude value None invalid, replacing by None (file S35M0782)


48 SMP profiles
       n                     first                      last  depth_max  no_gps
day                                                                            
DAY4  16 2026-03-06 20:27:08+00:00 2026-03-06 22:07:16+00:00     1700.0       1
Day1  26 2026-03-03 19:21:00+00:00 2026-03-03 21:45:57+00:00     1700.0       1
Day2   6 2026-03-04 17:02:44+00:00 2026-03-04 19:18:40+00:00     1700.0       0

Serials: {'35': 48}
Sample count range: 211389 - 411400


## 6. SnowScope CSV profiles

In [12]:
def read_snowscope(p: Path) -> tuple[dict, pd.DataFrame]:
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    meta, i = {}, 0
    while i < len(lines) and not lines[i].startswith("depth (mm)"):
        parts = lines[i].split(",", 1)
        if len(parts) == 2:
            meta[parts[0].strip()] = parts[1].strip()
        i += 1
    table = pd.read_csv(p, skiprows=i, na_values=["null"])
    return meta, table


ss_rows, ss_keys, ss_cols = [], Counter(), Counter()
for p in [p for p in files if classify(p) == "SnowScope"]:
    meta, t = read_snowscope(p)
    ss_keys.update(meta.keys()); ss_cols[tuple(t.columns)] += 1
    lat, lon = (meta.get("Location", ",").split(",") + [None])[:2]
    ss_rows.append({"file": p.name, "day": day_of(p)[:4], "serial": meta.get("serialNum"), "creator": meta.get("creator name"),
                    "time": meta.get("collectionTime"), "elev": pd.to_numeric(meta.get("elevation (m)"), errors="coerce"),
                    "lat": pd.to_numeric(lat, errors="coerce"), "lon": pd.to_numeric(lon, errors="coerce"),
                    "depth_mm": pd.to_numeric(meta.get("profileDepth (mm)"), errors="coerce"), "rows": len(t),
                    "hard_max": t.iloc[:, 1].max()})
ss = pd.DataFrame(ss_rows)
print(len(ss), "SnowScope profiles")
print(ss.groupby("day").agg(n=("file", "size"), serials=("serial", lambda s: sorted(s.dropna().unique())),
                            depth_min=("depth_mm", "min"), depth_max=("depth_mm", "max"), no_loc=("lat", lambda s: s.isna().sum())).to_string())
print("\nTable columns:", dict(ss_cols))
print("Creators:", ss.creator.value_counts().to_dict())

89 SnowScope profiles
       n         serials  depth_min  depth_max  no_loc
day                                                   
DAY3  10  [00249, 00374]       1638       1786       0
DAY4  27  [00249, 00374]        442       2462       0
DAY7  24         [00249]       2135       2387       0
Day1  21         [00374]       1542       1799       0
Day2   7         [00374]       1318       1764       0

Table columns: {('depth (mm)', 'hardness (kPa)', 'Unnamed: 2'): 89}
Creators: {'Benjamin Imbach': 53, 'Megan Cramb': 36}


In [13]:
v1_ss = sorted((V1 / "snowscope").rglob("*.csv"))
v1_keys, v1_cols = Counter(), Counter()
for p in v1_ss:
    meta, t = read_snowscope(p)
    v1_keys.update(meta.keys()); v1_cols[tuple(t.columns)] += 1
print("Header keys only in 2025-2026:", sorted(set(ss_keys) - set(v1_keys)))
print("Header keys only in v1:", sorted(set(v1_keys) - set(ss_keys)))
print("v1 table columns:", dict(v1_cols))
print("Optical reflectance column filled in 2025-2026:",
      sum(read_snowscope(p)[1].iloc[:, 2].notna().any() for p in files if classify(p) == "SnowScope" and read_snowscope(p)[1].shape[1] > 2))

Header keys only in 2025-2026: ['Max Profile Speed (m/s)', 'Profile Time (sec)']
Header keys only in v1: []
v1 table columns: {('depth (mm)', 'hardness (kPa)', 'optical Reflectance Avg'): 181, ('depth (mm)', 'hardness (kPa)', 'Unnamed: 2'): 138}


Optical reflectance column filled in 2025-2026: 0


## 7. K-band FMCW radar (`Radar_K`, `ka`)

Both folder names hold files from the same unit as v1 `radar_FMCW_K/`. The radar clock was not set in 2026:
dates come from the folder names.

In [14]:
def radar_k_header(p: Path) -> dict:
    h = {}
    for line in p.read_text(encoding="utf-8", errors="replace").splitlines()[:16]:
        if ":" in line and not line.startswith("#"):
            k, v = line.split(":", 1)
            h[k.strip()] = v.strip()
    return h


rk = []
for p in [p for p in files if classify(p) == "radar K-band" and p.suffix.lower() == ".txt"]:
    h = radar_k_header(p) if p.stat().st_size else {}
    rk.append({"file": rel(p), "day": day_of(p)[:4], "folder": "ka" if "ka" in p.parts else "Radar_K", "subfolder": p.parent.name,
               "size": p.stat().st_size, "radar_no": h.get("Radar No."), "date_in_file": h.get("Date"),
               "f_start": h.get("Start-Frequency [MHz]"), "f_stop": h.get("Stop-Frequency [MHz]"), "samples": h.get("Number of Samples")})
rk = pd.DataFrame(rk)
print(rk.groupby(["day", "folder", "subfolder"]).agg(n=("file", "size"), empty=("size", lambda s: (s == 0).sum()),
                                                      dates_in_file=("date_in_file", lambda s: sorted(s.dropna().unique()))).to_string())
print("\nUnits:", rk.radar_no.value_counts().to_dict(), "| bands:", rk[["f_start", "f_stop"]].drop_duplicates().values.tolist())
print("Empty files:", rk.loc[rk["size"] == 0, "file"].tolist())
v1h = radar_k_header(next((V1 / "radar_FMCW_K").rglob("*.txt")))
print("v1 unit:", v1h.get("Radar No."), v1h.get("Start-Frequency [MHz]"), v1h.get("Stop-Frequency [MHz]"))

                          n  empty dates_in_file
day  folder  subfolder                          
DAY4 ka      06_03_2026  23      0  [2025-09-03]
DAY7 Radar_K 10_03_2026  72      0  [2025-09-04]
DAY8 Radar_K 11_03_2026  18      0  [2025-09-04]
Day1 ka      03_03_2026  25      2  [2025-09-03]
Day2 ka      04_03_2026   3      0  [2025-09-03]

Units: {'2010000058': 139} | bands: [[nan, nan], ['23500', '26000']]
Empty files: ['Day1_RoundHill_03-03-2026/ka/03_03_2026/000020250903_1636.txt', 'Day1_RoundHill_03-03-2026/ka/03_03_2026/000120250903_1637.txt']
v1 unit: 2010000058 23500 26000


## 8. Dual Ku-band FMCW radar (`dku`) — new instrument

File name: `<counter>_<13|17>GHz_<site code>_<point>_<polarization>_<angle>deg.txt`.

In [15]:
NAME_RE = re.compile(r"^(\d{4})_(\d{2})GHz_([A-Za-z0-9]+)_(\d+)_([HV])_(\d+)deg\.txt$")


def dku_header(p: Path) -> dict:
    h, chirps = {}, 0
    with p.open(encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("# Chirp Number"):
                chirps += 1
            elif line.startswith("# ") and ":" in line and chirps == 0:
                k, v = line[2:].split(":", 1)
                h[k.strip()] = v.strip()
    h["chirps"] = chirps
    return h


dk = []
for p in [p for p in files if classify(p) == "radar dual-Ku"]:
    m = NAME_RE.match(p.name)
    h = dku_header(p) if p.suffix.lower() == ".txt" else {}
    dk.append({"file": rel(p), "day": day_of(p)[:4], "name_ok": bool(m), "counter": m and int(m.group(1)), "ghz": m and m.group(2),
               "site": m and m.group(3), "angle": m and int(m.group(6)), "pol": m and m.group(5),
               "ts": h.get("Timestamp"), "device": h.get("Device Number"), "fmin": h.get("Min Frequency"), "fmax": h.get("Max Frequency"),
               "hdr_site": h.get("Site Name"), "chirps": h.get("chirps")})
dk = pd.DataFrame(dk)
print(len(dk), "files;", (~dk.name_ok).sum(), "do not match the naming pattern:", dk.loc[~dk.name_ok, "file"].tolist())
ok = dk[dk.name_ok]
print(ok.groupby(["day", "site"]).agg(files=("file", "size"), counters=("counter", lambda s: f"{s.min()}-{s.max()}"),
                                      angles=("angle", lambda s: sorted(s.unique())), first=("ts", "min"), last=("ts", "max")).to_string())
print("\nDevices:", ok.groupby("ghz").device.unique().to_dict())
print("Bands (kHz):", ok.groupby("ghz")[["fmin", "fmax"]].first().to_dict("index"))
print("Chirps per file:", ok.chirps.value_counts().to_dict())
print("Header site != file-name site:", (ok.hdr_site != ok.site).sum())
pairs = ok.groupby(["day", "counter"]).ghz.nunique()
print("Counters without both 13 and 17 GHz files:", pairs[pairs < 2].index.tolist()[:20])

175 files; 0 do not match the naming pattern: []
           files counters                                      angles                       first                        last
day  site                                                                                                                    
DAY4 JBFP     24  833-844  [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]  2026-03-06T16:59:44.377022  2026-03-06T17:25:02.750323
     JBTR     30  847-862                                        [30]  2026-03-06T18:22:27.267069  2026-03-06T19:40:45.332606
DAY8 RHUT     36  863-880                                        [30]  2026-03-10T19:10:18.362294  2026-03-10T19:40:11.414278
Day1 RHFP     22  790-800  [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]  2026-03-03T19:06:21.614263  2026-03-03T19:19:46.193366
     RHTR     36  801-818                                        [30]  2026-03-03T19:36:21.645386  2026-03-03T20:06:07.436945
Day2 FIFP     22  819-829  [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

## 9. GNSS points (Emlid exports: KML, KMZ, CSV)

In [16]:
def kml_points(text: str) -> list[tuple]:
    pts = []
    for pm in re.findall(r"<Placemark>.*?</Placemark>", text, flags=re.S):
        name = re.search(r"<name>(.*?)</name>", pm, flags=re.S)
        c = re.search(r"<coordinates>\s*([-\d.]+),([-\d.]+),?([-\d.]*)", pm)
        if c:
            pts.append((name.group(1).strip() if name else "", float(c.group(2)), float(c.group(1)), float(c.group(3) or "nan")))
    return pts


gnss = []
for p in [p for p in files if classify(p) == "GNSS"]:
    if p.suffix.lower() == ".kml":
        pts = kml_points(p.read_text(encoding="utf-8", errors="replace"))
    elif p.suffix.lower() == ".kmz":
        z = zipfile.ZipFile(p)
        pts = kml_points(z.read(next(n for n in z.namelist() if n.endswith(".kml"))).decode("utf-8", errors="replace"))
    elif p.suffix.lower() == ".csv":
        df = pd.read_csv(p)
        pts = list(zip(df["Name"].astype(str), df["Latitude"], df["Longitude"], df["Ellipsoidal height"]))
    else:
        continue
    for n, lat, lon, h in pts:
        gnss.append({"file": rel(p), "day": day_of(p)[:4], "name": n, "lat": lat, "lon": lon, "h_ellips": h})
gnss = pd.DataFrame(gnss)
print(gnss.groupby("file").agg(points=("name", "size"), lat_min=("lat", "min"), lat_max=("lat", "max"),
                               lon_min=("lon", "min"), lon_max=("lon", "max"), h_min=("h_ellips", "min"), h_max=("h_ellips", "max")).to_string())

                                                                         points    lat_min    lat_max     lon_min     lon_max     h_min     h_max
file                                                                                                                                             
DAY4_JimBay-RoundHill-GopherButte_20260306/GPS_pts/JIM_BAY_TRANSECT.kml      19  51.234233  51.234671 -117.699283 -117.698653  1821.982  1841.738
DAY7_HermitMeadows_20260309/GPS_pts/HERMIT MEADOWS TR.csv                    24  51.330371  51.331085 -117.530873 -117.529957  2098.686  2109.571
DAY7_HermitMeadows_20260309/GPS_pts/HERMIT MEADOWS TR.kml                    24  51.330371  51.331085 -117.530873 -117.529957  2098.686  2109.571
DAY8_Fidelity_20260310/GPS/RH_2_TR.kml                                       19  51.235062  51.235263 -117.707804 -117.707495  2039.734  2044.397
Day1_RoundHill_03-03-2026/GPS_pts/RH_TR_2026.kmz                             19  51.235000  51.235310 -117.707400 -117.70700

In [17]:
csvs = [p for p in files if classify(p) == "GNSS" and p.suffix.lower() == ".csv"]
for p in csvs:
    df = pd.read_csv(p)
    print(p.name, "| solution:", df["Solution status"].value_counts().to_dict(), "| correction:", df["Correction type"].value_counts().to_dict(),
          "| receiver:", df.iloc[:, -2].unique().tolist() if df.shape[1] > 2 else None)
v1_cols = list(pd.read_csv(next((V1 / "spatial_reference").glob("*gps_points.csv"))).columns)
print("\nColumns only in 2025-2026 CSV:", sorted(set(pd.read_csv(csvs[0]).columns) - set(v1_cols)))
print("Columns only in v1 CSV:", sorted(set(v1_cols) - set(pd.read_csv(csvs[0]).columns)))

HERMIT MEADOWS TR.csv | solution: {'FIX': 24} | correction: {'RTK': 24} | receiver: ['Reach RS2']

Columns only in 2025-2026 CSV: ['Code description', 'Device serial number', 'Device type']
Columns only in v1 CSV: []


## 10. Scope summary

Site table, temporal coverage and file-derived extent. Coordinates come from GNSS exports, SMP headers
and SnowScope headers (WGS84, EPSG:4326). Heights: GNSS = ellipsoidal; SnowScope `elevation (m)` = phone/app value.

In [18]:
pts = pd.concat([
    gnss.assign(source="GNSS")[["day", "lat", "lon", "source"]],
    smp.dropna(subset=["lat"]).assign(source="SMP")[["day", "lat", "lon", "source"]],
    ss.dropna(subset=["lat"]).assign(source="SnowScope")[["day", "lat", "lon", "source"]],
])
# Outliers: points more than ~0.2 deg (~15-20 km) from the median of all RTK GNSS points, e.g. a phone
# location not refreshed in the SnowScope app. Listed, then excluded from the extent.
ref_lat, ref_lon = gnss.lat.median(), gnss.lon.median()
pts = pts.reset_index(drop=True)
pts["outlier"] = ((pts.lat - ref_lat).abs() > 0.2) | ((pts.lon - ref_lon).abs() > 0.2)
print("Location outliers (excluded from extent):")
print(pts[pts.outlier].groupby(["day", "source"]).agg(n=("lat", "size"), lat=("lat", "first"), lon=("lon", "first")).to_string())
print("\nSnowScope profiles per location cluster (rounded to 0.01 deg):")
print(ss.assign(cluster=ss.lat.round(2).astype(str) + ", " + ss.lon.round(2).astype(str))
        .groupby(["day", "cluster"]).agg(n=("file", "size"), elev_min=("elev", "min"), elev_max=("elev", "max")).round(1).to_string())
good = pts[~pts.outlier]
print()
print(good.groupby(["day", "source"]).agg(n=("lat", "size"), lat_min=("lat", "min"), lat_max=("lat", "max"),
                                          lon_min=("lon", "min"), lon_max=("lon", "max")).round(5).to_string())
bbox = dict(west=good.lon.min(), east=good.lon.max(), south=good.lat.min(), north=good.lat.max())
print("\nBounding box 2025-2026:", {k: round(float(v), 5) for k, v in bbox.items()})
h = gnss.h_ellips[gnss.h_ellips > 0]
print("GNSS points with height 0 (no height in export):", int((gnss.h_ellips == 0).sum()), "in", gnss.loc[gnss.h_ellips == 0, "file"].unique().tolist())
print("GNSS ellipsoidal height range (>0):", round(h.min(), 1), "-", round(h.max(), 1), "m")
ss_ok = ss.merge(good[good.source == "SnowScope"][["lat", "lon"]].drop_duplicates(), on=["lat", "lon"])
print("SnowScope app elevation range (non-outliers):", round(ss_ok.elev.min(), 1), "-", round(ss_ok.elev.max(), 1), "m")
print("Workbook ELEVATION fields:", hdr["ELEVATION"].dropna().astype(str).unique().tolist() if "ELEVATION" in hdr else None)

Location outliers (excluded from extent):
                n        lat         lon
day  source                             
DAY4 SnowScope  4  51.415519 -117.008748
Day2 SnowScope  3  51.415519 -117.008748

SnowScope profiles per location cluster (rounded to 0.01 deg):
                      n  elev_min  elev_max
day  cluster                               
DAY3 51.24, -117.71   6    2055.7    2058.7
     51.3, -117.52    4    1302.6    1302.6
DAY4 51.23, -117.7   12    1823.4    1872.8
     51.24, -117.7    5    1919.0    1923.0
     51.24, -117.71   6    2034.8    2037.6
     51.42, -117.01   4     805.7     805.7
DAY7 51.33, -117.53  24    2098.5    2111.2
Day1 51.24, -117.71  21    2023.7    2037.1
Day2 51.24, -117.7    4    1867.6    1867.6
     51.42, -117.01   3     805.7     805.7



                 n   lat_min   lat_max    lon_min    lon_max
day  source                                                 
DAY3 SnowScope  10  51.23527  51.30126 -117.70773 -117.51605
DAY4 GNSS       19  51.23423  51.23467 -117.69928 -117.69865
     SMP        15  51.23424  51.23467 -117.69933 -117.69868
     SnowScope  23  51.23421  51.23817 -117.70746 -117.69868
DAY7 GNSS       48  51.33037  51.33108 -117.53087 -117.52996
     SnowScope  24  51.33037  51.33105 -117.53089 -117.53000
DAY8 GNSS       19  51.23506  51.23526 -117.70780 -117.70750
Day1 GNSS       19  51.23500  51.23531 -117.70740 -117.70700
     SMP        25  51.23502  51.23531 -117.70747 -117.70702
     SnowScope  21  51.23500  51.23534 -117.70743 -117.70704
Day2 GNSS        3  51.23648  51.23648 -117.70082 -117.70080
     SMP         6  51.23645  51.23649 -117.70088 -117.70071
     SnowScope   4  51.23649  51.23650 -117.70089 -117.70088

Bounding box 2025-2026: {'west': -117.7078, 'east': -117.51605, 'south': 51.23421, '

In [19]:
dates = pd.concat(ignore_index=True, objs=[
    smp.assign(date=smp.timestamp.dt.date.astype(str), src="SMP")[["day", "date", "src"]],
    ok.assign(date=ok.ts.str[:10], src="dual-Ku")[["day", "date", "src"]],
    pd.DataFrame([{"day": r["day"], "date": d, "src": "IRIS"} for r in iris for d in (r["dates"] or [])]),
])
print(pd.crosstab([dates.day, dates.date], dates.src).to_string())
print("\nDay folders vs nominal dates:", DAY_DATES)

src              IRIS  SMP  dual-Ku
day  date                          
DAY3 2026-3-5       1    0        0
DAY4 2026-03-06     0   16       52
     2026-3-6       1    0        0
DAY7 2026-3-9       1    0        0
DAY8 2026-03-10     0    0       36
Day1 2026-03-03     0   26       58
     2026-3-3       2    0        0
Day2 2026-03-04     0    6       24
     2026-3-4       1    0        0

Day folders vs nominal dates: {'Day1_RoundHill_03-03-2026': '2026-03-03', 'Day2_Fidelity_RH_04-03-2026': '2026-03-04', 'DAY3_RoundHill Drone and Rogers Pass_05-03-2026': '2026-03-05', 'DAY4_JimBay-RoundHill-GopherButte_20260306': '2026-03-06', 'DAY5_RestDay_20260307': '2026-03-07', 'DAY6_ReposeEtFilm_20260308': '2026-03-08', 'DAY7_HermitMeadows_20260309': '2026-03-09', 'DAY8_Fidelity_20260310': '2026-03-10'}


## 11. Cross-period comparison with published v1

Instrument folders, formats and conventions in v1 (`published/v1/`) vs 2025-2026 raw files.

In [20]:
v1_classes = {d.name: len([f for f in d.rglob("*") if f.is_file()]) for d in sorted(V1.iterdir()) if d.is_dir()}
new_classes = inv[~inv.cls.str.contains("excluded")].cls.value_counts().to_dict()
print("v1 deposit folders:", v1_classes)
print("2025-2026 in-scope classes:", new_classes)
v1_ext = Counter(f.suffix.lower() for f in V1.rglob("*") if f.is_file())
print("\nv1 extensions:", dict(v1_ext))
print("2025-2026 in-scope extensions:", inv[~inv.cls.str.contains("excluded")].ext.value_counts().to_dict())

v1 deposit folders:

 {'documentation': 8, 'iris': 6, 'radar_FMCW_K': 166, 'snow_stratigraphy': 7, 'snowmicropenetrometer': 79, 'snowscope': 319, 'spatial_reference': 10}
2025-2026 in-scope classes: {'radar dual-Ku': 175, 'radar K-band': 141, 'SnowScope': 89, 'SMP': 48, 'stratigraphy workbook': 14, 'IRIS': 13, 'field readme / notes': 8, 'GNSS': 6, 'spatial linkage workbook': 4}

v1 extensions: {'.txt': 184, '.xlsx': 11, '.csv': 322, '.zip': 3, '.pnt': 79}
2025-2026 in-scope extensions: {'.txt': 329, '.csv': 90, '.pnt': 48, '.xlsx': 18, '.docx': 8, '.kml': 4, '.kmz': 1}


## 12. Synthesis

Key counts used in `artifacts/2025-2026/data_exploration.md`.

In [21]:
summary = {
    "files total": len(inv),
    "in scope (non-excluded classes)": int((~inv.cls.str.contains("excluded")).sum()),
    "stratigraphy workbooks": len(strat),
    "SMP profiles": len(smp),
    "SnowScope profiles": len(ss),
    "IRIS files": len(iris),
    "K-band radar files": len(rk),
    "K-band empty files": int((rk["size"] == 0).sum()),
    "dual-Ku files": len(dk),
    "GNSS points": len(gnss),
}
for k, v in summary.items():
    print(f"{k:35s} {v}")

files total                         567
in scope (non-excluded classes)     498
stratigraphy workbooks              14
SMP profiles                        48
SnowScope profiles                  89
IRIS files                          13
K-band radar files                  141
K-band empty files                  2
dual-Ku files                       175
GNSS points                         108
